Day Ahead Data

Source Format: XML
Report frequency: Daily
Data granularity: 24 hourly prices per delivery day

In [8]:
import requests 

url = "https://reports-public.ieso.ca/public/DAHourlyOntarioZonalPrice/PUB_DAHourlyOntarioZonalPrice_20261015.xml"
response = requests.get(url)

print(response.status_code)
print(response.text[:1000])

200
<?xml version="1.0" encoding="UTF-8"?>
<?xml-stylesheet type="text/xsl" href="https://reports-public.ieso.ca/docrefs/stylesheet/DAHourlyOntarioZonalPrice_HTML_t1-1.xsl" ?>
<Document docID="DAHourlyOntarioZonalPrice" xmlns="http://www.ieso.ca/schema" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.ieso.ca/schema https://reports-public.ieso.ca/docrefs/schema/DAHourlyOntarioZonalPrice_r1.xsd">
<DocHeader>
<DocTitle>Day Ahead Hourly Ontario Zonal Energy Price Report</DocTitle>
<DocRevision>1</DocRevision>
<DocConfidentiality>
<DocConfClass>PUB</DocConfClass>
</DocConfidentiality>
<CreatedAt>2026-10-14T12:31:28</CreatedAt>
</DocHeader>
<DocBody>
<DeliveryDate>2026-10-15</DeliveryDate>
<HourlyPriceComponents>
<PricingHour>1</PricingHour>
<ZonalPrice>45.15</ZonalPrice>
<LossPriceCapped>0.31</LossPriceCapped>
<CongestionPriceCapped>0</CongestionPriceCapped>
<Flag>DSO-RD</Flag>
</HourlyPriceComponents>
<HourlyPriceComponents>
<PricingHour>2</PricingHour>

In [39]:
import xml.etree.ElementTree as ET

root = ET.fromstring(response.text)

hourly_price_records = []

for child in root:
    if child.tag.endswith("DocBody"):
        for item in child:
            if item.tag.endswith("DeliveryDate"):
                delivery_date = item.text
            
            elif item.tag.endswith("HourlyPriceComponents"):
                hourly_prices = {"DeliveryDate": delivery_date}

                for value in item:
                    field = value.tag.split("}")[1]
                    if field == "PricingHour":
                        hourly_prices[value.tag.split("}")[1]] = int(value.text)
                    elif field == "ZonalPrice" or value.tag.split("}")[1] == "LossPriceCapped" or value.tag.split("}")[1] == "CongestionPriceCapped":
                        hourly_prices[value.tag.split("}")[1]] = float(value.text)
                    else:
                        hourly_prices[value.tag.split("}")[1]] = value.text    
                
                hourly_price_records.append(hourly_prices)

print(hourly_price_records)
print(len(hourly_price_records))

[{'DeliveryDate': '2026-10-15', 'PricingHour': 1, 'ZonalPrice': 45.15, 'LossPriceCapped': 0.31, 'CongestionPriceCapped': 0.0, 'Flag': 'DSO-RD'}, {'DeliveryDate': '2026-10-15', 'PricingHour': 2, 'ZonalPrice': 46.8, 'LossPriceCapped': 0.41, 'CongestionPriceCapped': 0.0, 'Flag': 'DSO-RD'}, {'DeliveryDate': '2026-10-15', 'PricingHour': 3, 'ZonalPrice': 44.54, 'LossPriceCapped': 0.37, 'CongestionPriceCapped': 0.0, 'Flag': 'DSO-RD'}, {'DeliveryDate': '2026-10-15', 'PricingHour': 4, 'ZonalPrice': 43.22, 'LossPriceCapped': 0.34, 'CongestionPriceCapped': 0.0, 'Flag': 'DSO-RD'}, {'DeliveryDate': '2026-10-15', 'PricingHour': 5, 'ZonalPrice': 47.52, 'LossPriceCapped': 0.27, 'CongestionPriceCapped': 0.0, 'Flag': 'DSO-RD'}, {'DeliveryDate': '2026-10-15', 'PricingHour': 6, 'ZonalPrice': 61.33, 'LossPriceCapped': 0.15, 'CongestionPriceCapped': 0.0, 'Flag': 'DSO-RD'}, {'DeliveryDate': '2026-10-15', 'PricingHour': 7, 'ZonalPrice': 87.99, 'LossPriceCapped': 0.05, 'CongestionPriceCapped': 0.19, 'Flag': 'D

In [31]:
import pandas as pd

df = pd.DataFrame(hourly_price_records)
df["DeliveryDate"] = pd.to_datetime(df["DeliveryDate"])

df.head()

,DeliveryDate,PricingHour,ZonalPrice,LossPriceCapped,CongestionPriceCapped,Flag
0,2026-10-15,1,45.15,0.31,0.0,DSO-RD
1,2026-10-15,2,46.80,0.41,0.0,DSO-RD
2,2026-10-15,3,44.54,0.37,0.0,DSO-RD
3,2026-10-15,4,43.22,0.34,0.0,DSO-RD
4,2026-10-15,5,47.52,0.27,0.0,DSO-RD
